<a href="https://colab.research.google.com/github/ZihanEvelyn/Faceplusplus-test/blob/main/Face%2B%2B12sample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from pathlib import Path
from io import BytesIO
import time
import pandas as pd
import requests
from google.colab import files, userdata
from PIL import Image, ImageOps

uploaded_book = files.upload()
if len(uploaded_book) != 1:
    raise ValueError("Select one Excel workbook.")
book = next(iter(uploaded_book.values()))
articles = pd.read_excel(BytesIO(book), sheet_name="候选报道")
photos = pd.read_excel(BytesIO(book), sheet_name="逐图核验")
def prepare_sheet(frame, columns, sheet_name):
    table = frame[columns].copy()
    table = table.apply(lambda col: col.map(lambda v: (v.strip() or pd.NA) if isinstance(v, str) else v))
    table = table.dropna(how="all")
    numbers = pd.to_numeric(table["顺序"], errors="coerce")
    bad = numbers.isna() | (numbers < 1) | (numbers % 1 != 0)
    if bad.any():
        display(table.loc[bad])
        raise ValueError(f"请检查「{sheet_name}」上方列出的行：顺序须为正整数。")
    table["顺序"] = numbers.astype(int)
    return table

article_cols = ["顺序", "Reuters 标题", "原文 URL", "筛选结果"]
article_index = prepare_sheet(articles, article_cols, "候选报道").drop_duplicates()
conflicts = article_index["顺序"].duplicated(keep=False)
if conflicts.any():
    display(article_index.loc[conflicts].sort_values("顺序"))
    raise ValueError("同一报道顺序对应不同记录。请对照原文核对上方记录及「逐图核验」中的对应顺序。")
photo_index = prepare_sheet(photos, ["顺序", "照片序号", "图片类型", "可见人脸", "画面或核验情况"], "逐图核验")
unmatched = ~photo_index["顺序"].isin(article_index["顺序"])
if unmatched.any():
    display(photo_index.loc[unmatched])
    raise ValueError("上方照片的报道顺序在「候选报道」中找不到。")
manifest = photo_index.merge(article_index, on="顺序", how="left", validate="many_to_one")
def image_id(row):
    prefix = f"N{int(row['顺序']):02d}"
    if row["图片类型"] == "视频封面":
        return prefix + "_cover"
    return f"{prefix}_P{int(row['照片序号']):02d}"

manifest.insert(0, "image_id", manifest.apply(image_id, axis=1))
assert manifest["image_id"].is_unique
display(manifest[["image_id", "顺序", "照片序号", "图片类型", "可见人脸", "画面或核验情况", "原文 URL"]])

Saving Reuters_12_Colab.xlsx to Reuters_12_Colab.xlsx


,image_id,顺序,照片序号,图片类型,可见人脸,画面或核验情况,原文 URL
0,N01_P01,1,1,新闻照片,是,车手近景,https://www.reuters.com/sports/formula1/russel...
1,N01_P02,1,2,新闻照片,是,车手戴帽正面,https://www.reuters.com/sports/formula1/russel...
2,N01_P03,1,3,新闻照片,是,车手侧脸,https://www.reuters.com/sports/formula1/russel...
3,N02_P01,2,1,新闻照片,是,讲台前的发言人,https://www.reuters.com/sports/former-ioc-vice...
4,N03_P01,3,1,新闻照片,是,两人握手，侧脸可见,https://www.reuters.com/world/europe/trump-say...
5,N04_P01,4,1,新闻照片,是,发言人近景,https://www.reuters.com/business/energy/co-lea...
6,N04_P02,4,2,新闻照片,是,户外人物近景,https://www.reuters.com/business/energy/co-lea...
7,N04_P03,4,3,新闻照片,是,两人会谈,https://www.reuters.com/business/energy/co-lea...
8,N05_P01,5,1,新闻照片,是,采访现场,https://www.reuters.com/world/frances-problems...
9,N05_P02,5,2,新闻照片,是,西装人物坐在桌前,https://www.reuters.com/world/frances-problems...


In [5]:
uploaded_images = files.upload()
images = {}
for name, data in uploaded_images.items():
    names = {v.upper(): v for v in manifest["image_id"]}
    key = names.get(Path(name).stem.upper())
    if key not in set(manifest["image_id"]):
        raise ValueError(f"Unknown image ID: {name}. Check the names in step 1.")
    if key in images:
        raise ValueError(f"Two files match {key}. Upload only one.")
    images[key] = (name, data)
print(f"Matched {len(images)} image(s).")

Saving N01_P01.jpg to N01_P01.jpg
Saving N01_P02.jpg to N01_P02.jpg
Saving N01_P03.jpg to N01_P03.jpg
Saving N02_P01.jpg to N02_P01.jpg
Saving N03_P01.jpg to N03_P01.jpg
Saving N04_P01.jpg to N04_P01.jpg
Saving N04_P02.jpg to N04_P02.jpg
Saving N04_P03.jpg to N04_P03.jpg
Saving N05_P01.jpg to N05_P01.jpg
Saving N06_cover.jpg to N06_cover.jpg
Saving N06_P01.jpg to N06_P01.jpg
Saving N07_P01.jpg to N07_P01.jpg
Saving N08_P01.jpg to N08_P01.jpg
Saving N08_P02.jpg to N08_P02.jpg
Saving N09_cover.jpg to N09_cover.jpg
Saving N09_P01.jpg to N09_P01.jpg
Saving N10_P01.jpg to N10_P01.jpg
Saving N11_P01.jpg to N11_P01.jpg
Saving N12_P01.jpg to N12_P01.jpg
Matched 19 image(s).


In [6]:
api_key = userdata.get("FACEPP_API_KEY").strip()
api_secret = userdata.get("FACEPP_API_SECRET").strip()
emotions = ["happiness", "sadness", "anger", "disgust", "fear", "surprise", "neutral"]
endpoint = "https://api-us.faceplusplus.com/facepp/v3/detect"

def detect(raw):
    with Image.open(BytesIO(raw)) as original:
        img = ImageOps.exif_transpose(original).convert("RGB")
        img.thumbnail((2000, 2000))
        buf = BytesIO()
        img.save(buf, format="JPEG", quality=88)
    for attempt in range(3):
        response = requests.post(endpoint,
            data={"api_key": api_key, "api_secret": api_secret, "return_attributes": "emotion"},
            files={"image_file": ("photo.jpg", buf.getvalue(), "image/jpeg")}, timeout=60)
        result = response.json()
        error = result.get("error_message", "")
        if response.ok and not error and isinstance(result.get("faces"), list):
            return result["faces"], img.size
        if attempt < 2 and (response.status_code == 429 or response.status_code >= 500 or "CONCURRENCY" in error):
            time.sleep(2 ** (attempt + 1))
            continue
        raise ValueError(error or f"HTTP {response.status_code}")

image_rows, face_rows = [], []
for item in manifest.to_dict("records"):
    key = item["image_id"]
    summary = {**item, "filename": None, "status": "not_uploaded",
               "faces_detected": None, "faces_scored": None, "sent_width": None, "sent_height": None,
               **{"mean_" + e: None for e in emotions}, "error": None}
    image_rows.append(summary)
    if key not in images:
        continue
    filename, raw = images[key]
    summary["filename"] = filename
    try:
        faces, size = detect(raw)
        complete = []
        for number, face in enumerate(faces, start=1):
            scores = face.get("attributes", {}).get("emotion") or {}
            values = {e: scores.get(e) for e in emotions}
            if all(isinstance(v, (int, float)) for v in values.values()):
                complete.append(values)
            face_rows.append({"image_id": key, "face_number": number,
                              **face.get("face_rectangle", {}), **values})
        summary.update(faces_detected=len(faces), faces_scored=len(complete),
                       sent_width=size[0], sent_height=size[1])
        if complete:
            for e in emotions:
                summary["mean_" + e] = sum(v[e] for v in complete) / len(complete)
        summary["status"] = ("no_face" if not faces else
                             "no_scores" if not complete else
                             "partial_scores" if len(complete) < len(faces) else "ok")
    except (requests.RequestException, ValueError, OSError) as exc:
        summary.update(status="error", error=str(exc))
    print(key, summary["status"], summary["error"] or "")
    time.sleep(1)

image_results = pd.DataFrame(image_rows)
face_results = pd.DataFrame(face_rows, columns=["image_id", "face_number", "left", "top",
                                                "width", "height", *emotions])
display(image_results.loc[image_results.filename.notna(),
        ["image_id", "可见人脸", "status", "faces_detected", "faces_scored", "mean_happiness", "error"]])

N01_P01 ok 
N01_P02 ok 
N01_P03 ok 
N02_P01 ok 
N03_P01 ok 
N04_P01 ok 
N04_P02 ok 
N04_P03 ok 
N05_P01 ok 
N06_cover ok 
N06_P01 ok 
N07_P01 ok 
N08_P01 no_face 
N08_P02 ok 
N09_cover ok 
N09_P01 ok 
N10_P01 partial_scores 
N11_P01 no_face 
N12_P01 ok 


,image_id,可见人脸,status,faces_detected,faces_scored,mean_happiness,error
0,N01_P01,是,ok,1.0,1.0,88.983000,None
1,N01_P02,是,ok,1.0,1.0,0.000000,None
2,N01_P03,是,ok,2.0,2.0,53.798000,None
3,N02_P01,是,ok,1.0,1.0,48.670000,None
4,N03_P01,是,ok,2.0,2.0,50.007000,None
5,N04_P01,是,ok,1.0,1.0,0.049000,None
6,N04_P02,是,ok,1.0,1.0,98.484000,None
7,N04_P03,是,ok,2.0,2.0,11.166500,None
8,N05_P01,是,ok,2.0,2.0,97.546000,None
13,N06_cover,是,ok,1.0,1.0,0.187000,None


In [7]:
output = "Reuters_FacePP_results.xlsx"
with pd.ExcelWriter(output, engine="openpyxl") as writer:
    image_results.to_excel(writer, sheet_name="images", index=False)
    face_results.to_excel(writer, sheet_name="faces", index=False)
    for sheet in writer.sheets.values():
        sheet.freeze_panes = "B2"
        sheet.auto_filter.ref = sheet.dimensions
files.download(output)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>